# North Atlantic surface density flux and deep convection — alternate spin-ups vs Full-CPL

Why the AMOC differs between the runs: the surface density flux that drives deep-water formation in the Labrador Sea,
Irminger Sea, GIN Seas and the subpolar North Atlantic, split into its thermal and haline parts, with March
mixed-layer depth (deep convection) and the AMOC at 26°N (from `3_tke_amoc_compare_analysis`).

Per decade, from the remapped 0.5° decadal climatologies `post/time_series/ocn_2d` (annual for the fluxes, March for
the mixed layer), area-weighted over the ocean cells of each region box (`util/natl_density_flux.REGIONS`):
* F_T = −(α/c_p) Q_net, F_S = −β S F_fw/(1 − S/1000) + β·1000·F_salt (TEOS-10 α, β at the surface; Q_net, F_fw,
  F_salt = net heat, freshwater and sea-ice salt flux into the ocean); F_ρ = F_T + F_S > 0 makes surface water denser
* March mixed-layer depth (density threshold): regional mean and the area deeper than `CONV_DEPTH`

Note: the boxes include the fresh shelves (e.g. the Labrador shelf), which lower the regional mean SSS. FC's decadal
climatologies end at year 250 (FC-FOSI: 350) and the piControl has none, so there is no significance band.

Figures:
* `natl_density_flux_<years>.pdf` — rows = regions; columns = F_ρ with F_T (dashed) and F_S (dotted), March MLD, SSS
* `natl_convection_amoc_<years>.pdf` — convective area per region and AMOC at 26°N, decadal means

Figures go to `FIG_DIR` (public_html), regional decadal series to `DATA_DIR`.

In [ ]:
import os, sys
WF_ROOT = os.path.abspath("..")            # coupled_spinup_eval/
if WF_ROOT not in sys.path:
    sys.path.insert(0, WF_ROOT)
os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")
%matplotlib inline

import matplotlib.pyplot as plt

import numpy as np
import xarray as xr

from util.experiments import make_run_dict
from util.figures import publish
from util.natl_density_flux import REGIONS, regional_decades, plot_region_panels

## Setup

In [ ]:
# ---- paths ----
MODEL_ROOT   = "/lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/E3SMv3_testings"     # case directories
OUTPUT_ROOT  = "/lcrc/group/e3sm/public_html/diagnostic_output/ac.szhang/v3spinup_paper"   # published figures/ and data/
OUTPUT_URL   = "https://web.lcrc.anl.gov/public/e3sm/diagnostic_output/ac.szhang/v3spinup_paper"
FIG_DIR      = f"{OUTPUT_ROOT}/figures/natl_density_flux"
DATA_DIR     = f"{OUTPUT_ROOT}/data/natl_density_flux"     # cached/derived data (index: data/README.md)

CLIMO_SUBDIR = "post/time_series/ocn_2d"                   # decadal climatologies under MODEL_ROOT/<case>/
AMOC_DIR     = f"{OUTPUT_ROOT}/data/tke_amoc"              # AMOC26 series of 3_tke_amoc_compare_analysis

# ---- experiments: case directory names (under MODEL_ROOT) and model-year periods ----
EXPERIMENTS = {
    "Full-CPL": {
        "spin-up":   {"name": "20231209.v3.LR.piControl-spinup.chrysalis",      "period": "0001-2000"},
        "piControl": {"name": "v3.LR.piControl",                                "period": "0001-0500"},
    },
    "AltBG1": {
        "spin-up":   {"name": "20240707.v3.LR.piControl.Rerun.GC2BC.chrysalis", "period": "0001-0220"},
        "piControl": {"name": "20240707.v3.LR.piControl.Rerun.GC2BC.chrysalis", "period": "0221-0350"},
    },
    "AltBG2": {
        "spin-up":   {"name": "20240818.v3.LR.piControl.Rerun.GC2BC.chrysalis", "period": "0001-0220"},
        "piControl": {"name": "20240818.v3.LR.piControl.Rerun.GC2BC.chrysalis", "period": "0221-0350"},
    },
}
LABEL = {"Full-CPL": "FC", "AltBG1": "FC-FOSI-S1", "AltBG2": "FC-FOSI-S2", "AltBG3": "FC-FOSI-S3"}

# ---- analysis settings ----
EXPS        = ["Full-CPL", "AltBG1", "AltBG2"]   # runs shown
COLORS      = ["black", "#cc0000", "#377eb8"]    # one per run
YEARS       = {"Full-CPL": (1, 250), "AltBG1": (1, 350), "AltBG2": (1, 350)}   # decades available on disk
SWITCH_YEAR = 221           # first fully coupled year of the FC-FOSI runs
FOSI_SEGMENTS = {           # FOSI (data-atmosphere) years of each FC-FOSI run
    "AltBG1": [(21, 80), (91, 150), (161, 220)],
    "AltBG2": [(61, 220)],
}
CONV_DEPTH  = 500.0         # m, March mixed layer deeper than this counts as convective area

In [ ]:
# Shared inputs/outputs
run_dict = make_run_dict(EXPERIMENTS, MODEL_ROOT)
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
publish(OUTPUT_ROOT, FIG_DIR)
print("figures ->", FIG_DIR)
print("data    ->", DATA_DIR)
print("web     ->", FIG_DIR.replace(OUTPUT_ROOT, OUTPUT_URL))

## 1 Regional decadal series
Cached per run as `natl_decadal_<exp>_<years>.nc` (~18 s per climatology file on first use; `FORCE = True` rebuilds).

In [ ]:
FORCE = False
reg = {}
for t in EXPS:
    yrs = YEARS[t]
    cache = os.path.join(DATA_DIR, f"natl_decadal_{t}_{yrs[0]:04d}-{yrs[1]:04d}.nc")
    if os.path.isfile(cache) and not FORCE:
        reg[t] = xr.open_dataset(cache).load()
    else:
        case = run_dict[t]["spin-up"]["name"]
        reg[t] = regional_decades(os.path.join(MODEL_ROOT, case, CLIMO_SUBDIR), case, yrs, conv_depth=CONV_DEPTH, verbose=False)
        reg[t].attrs.update(case=case, conv_depth_m=CONV_DEPTH)
        reg[t].to_netcdf(cache)
    print(f"{LABEL[t]:11s} {reg[t].sizes['decade']} decades")

# AMOC at 26N: decadal means of the monthly series of 3_tke_amoc_compare_analysis
amoc = {}
for t in EXPS:
    with xr.open_dataset(os.path.join(AMOC_DIR, f"ocean_surface_tke_amoc_Global_ts_yr0001-0350_ts_{t}.nc")) as ds:
        yr = np.floor(ds["time_year"].values).astype(int)
        s = xr.DataArray(ds["AMOC26"].values, dims="month", coords={"decade": ("month", 10 * ((yr - 1) // 10) + 5.5)})
    amoc[t] = s.groupby("decade").mean()
publish(DATA_DIR)

## 2 Figures
### Settings
Density fluxes are drawn in 10⁻⁶ kg m⁻² s⁻¹.

In [ ]:
COLUMNS = [
    (r"F$_\rho$ (solid), F$_T$ (dashed), F$_S$ (dotted)", [("F_rho", 1e6, "-"), ("F_T", 1e6, "--"), ("F_S", 1e6, ":")],
     r"10$^{-6}$ kg m$^{-2}$ s$^{-1}$"),
    ("March mixed-layer depth", [("MLD_mean", 1.0, "-")], "m"),
    ("SSS", [("SSS", 1.0, "-")], "psu"),
]
PANEL_STYLE = dict(fontz=11.0)

### Plot

In [ ]:
results = {LABEL[t]: dict(ds=reg[t], color=c) for t, c in zip(EXPS, COLORS)}
segments = {LABEL[t]: FOSI_SEGMENTS.get(t, []) for t in EXPS}
span = f"{min(y[0] for y in YEARS.values()):04d}-{max(y[1] for y in YEARS.values()):04d}"
out = os.path.join(FIG_DIR, f"natl_density_flux_{span}.pdf")
fig, axes = plot_region_panels(results, COLUMNS, segments=segments, markers=[SWITCH_YEAR], **PANEL_STYLE, save=out)
plt.show()
print("web:", out.replace(OUTPUT_ROOT, OUTPUT_URL))

regions = list(REGIONS)
fig, axes = plt.subplots(1, len(regions) + 1, figsize=(4.2 * (len(regions) + 1), 3.2), layout="constrained")
for ax, r in zip(axes, regions):
    for t, c in zip(EXPS, COLORS):
        ax.plot(reg[t]["decade"], reg[t]["conv_area"].sel(region=r), "o-", color=c, ms=3, label=LABEL[t])
    ax.set_title(f"{r}: area MLD > {CONV_DEPTH:.0f} m", fontsize=10, loc="left"); ax.set_ylabel(r"10$^3$ km$^2$")
for t, c in zip(EXPS, COLORS):
    axes[-1].plot(amoc[t]["decade"], amoc[t], "o-", color=c, ms=3, label=LABEL[t])
axes[-1].set_title("AMOC at 26°N (decadal mean)", fontsize=10, loc="left"); axes[-1].set_ylabel("Sv")
for ax in axes:
    ax.axvline(SWITCH_YEAR, color="0.4", ls=":", lw=1); ax.grid(alpha=0.3); ax.set_xlabel("Model year")
axes[-1].legend(frameon=False, fontsize=9)
out = os.path.join(FIG_DIR, f"natl_convection_amoc_{span}.pdf")
fig.savefig(out, dpi=300, bbox_inches="tight")
plt.show()

publish(FIG_DIR, DATA_DIR)
print("web:", out.replace(OUTPUT_ROOT, OUTPUT_URL))